In [1]:
!nvidia-smi
!pip install -q -U bitsandbytes transformers peft accelerate datasets scikit-learn

Mon Sep 28 16:00:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
import json
import os
import random
import re
import subprocess
import time
import zipfile

import numpy as np
import pandas as pd
import torch

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
N_TRAIN, N_VAL, N_TEST = 3000, 200, 300
MAX_NARRATIVE_CHARS = 1000
MAX_LEN = 900
NUM_EPOCHS = 2
LORA_R = 16
USE_MLFLOW = False

CSV_PATH = None
ARCHIVE_URL = (
    "https://files.consumerfinance.gov/f/documents/"
    "CCDB_Export_4_November_2022_through_August_2023.zip"
)
ZIP_PATH = "/content/ccdb_archive.zip"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


In [7]:
from typing import Sequence

PRODUCT_ALIASES = (
    ("credit reporting", "Credit reporting"),
    ("credit card", "Credit card or prepaid card"),
    ("prepaid card", "Credit card or prepaid card"),
    ("bank account", "Checking or savings account"),
    ("checking or savings", "Checking or savings account"),
    ("money transfer", "Money transfer or virtual currency"),
)


def find_column(columns: Sequence[str], name: str) -> str:
    normalized = {c: c.lower().replace("_", " ").strip() for c in columns}
    for col, norm in normalized.items():
        if norm == name:
            return col
    for col, norm in normalized.items():
        if name in norm:
            return col
    raise KeyError(f"No column matching '{name}' in {list(columns)}")


def normalize_product(product: str) -> str:
    lowered = product.lower()
    for prefix, canonical in PRODUCT_ALIASES:
        if lowered.startswith(prefix):
            return canonical
    return product


def fetch_archive(url: str, dest: str) -> str:
    if os.path.exists(dest):
        return dest
    print("Downloading CFPB archive...")
    try:
        subprocess.run(["wget", "-q", "-O", dest, url], check=True)
    except (subprocess.CalledProcessError, FileNotFoundError) as exc:
        if os.path.exists(dest):
            os.remove(dest)
        raise RuntimeError(
            "Download failed. Upload the zip to Colab manually and set CSV_PATH."
        ) from exc
    return dest


def open_source(path: str):
    if not path.lower().endswith(".zip"):
        return open(path, "rb")
    archive = zipfile.ZipFile(path)
    members = [n for n in archive.namelist() if n.lower().endswith(".csv")]
    if not members:
        raise RuntimeError(f"No CSV found in archive. Contents: {archive.namelist()}")
    return archive.open(members[0])


def load_narratives(
    path: str, max_rows: int = 150_000, chunk_size: int = 200_000
) -> pd.DataFrame:
    with open_source(path) as handle:
        header = pd.read_csv(handle, nrows=0).columns

    source_cols = {name: find_column(header, name) for name in ("product", "issue", "narrative")}
    print("Columns:", source_cols)
    rename_map = {
        source_cols["product"]: "product",
        source_cols["issue"]: "issue",
        source_cols["narrative"]: "text",
    }

    frames, total = [], 0
    with open_source(path) as handle:
        reader = pd.read_csv(
            handle,
            usecols=list(source_cols.values()),
            chunksize=chunk_size,
            encoding_errors="replace",
        )
        for chunk in reader:
            chunk = chunk.rename(columns=rename_map).dropna()
            chunk = chunk[chunk["text"].str.len() > 200].copy()
            chunk["text"] = (
                chunk["text"]
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
                .str.slice(0, MAX_NARRATIVE_CHARS)
            )
            frames.append(chunk)
            total += len(chunk)
            if total >= max_rows:
                break

    if not frames or total == 0:
        raise RuntimeError("No complaints with narratives found. Try a different archive file.")
    return pd.concat(frames, ignore_index=True)


def build_splits(
    df: pd.DataFrame, n_train: int, n_val: int, n_test: int, seed: int
):
    need = n_train + n_val + n_test
    df = df.assign(product=df["product"].map(normalize_product))
    df = df[df["text"].str.len() >= 100]

    products = df["product"].value_counts().head(6).index.tolist()
    df = df[df["product"].isin(products)].sample(frac=1, random_state=seed)
    df = df.groupby("product", group_keys=False).head(max(need // 4, 1))

    top_issues = df["issue"].value_counts().head(12).index.tolist()
    df = df.assign(issue=df["issue"].where(df["issue"].isin(top_issues), "Other"))
    issues = top_issues + ["Other"]

    df = df.sample(frac=1, random_state=seed).reset_index(drop=True).iloc[:need]
    if len(df) < need:
        print(f"WARNING: only {len(df)} usable rows, expected {need}")

    train = df.iloc[:n_train].reset_index(drop=True)
    val = df.iloc[n_train : n_train + n_val].reset_index(drop=True)
    test = df.iloc[n_train + n_val :].reset_index(drop=True)
    return train, val, test, products, issues


source = CSV_PATH or fetch_archive(ARCHIVE_URL, ZIP_PATH)
raw = load_narratives(source)
print(f"Loaded {len(raw):,} complaints with narratives")

train_df, val_df, test_df, PRODUCTS, ISSUES = build_splits(
    raw, N_TRAIN, N_VAL, N_TEST, SEED
)

print(f"Split sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
print("PRODUCTS:", PRODUCTS)
print("ISSUES:", ISSUES)
print(train_df["product"].value_counts(normalize=True).round(3))
print(train_df.iloc[0].to_dict())

Columns: {'product': 'Product', 'issue': 'Issue', 'narrative': 'Consumer complaint narrative'}
Loaded 205,827 complaints with narratives
Split sizes: train=3000, val=200, test=300
PRODUCTS: ['Credit reporting', 'Debt collection', 'Credit card or prepaid card', 'Checking or savings account', 'Mortgage', 'Money transfer or virtual currency']
ISSUES: ['Managing an account', 'Trouble during payment process', 'Attempts to collect debt not owed', 'Incorrect information on your report', "Problem with a credit reporting company's investigation into an existing problem", 'Fraud or scam', 'Improper use of your report', 'Problem with a purchase shown on your statement', 'Struggling to pay mortgage', 'Other transaction problem', 'Written notification about debt', 'Problem with a lender or other company charging your account', 'Other']
product
Credit reporting                      0.170
Checking or savings account           0.168
Money transfer or virtual currency    0.168
Credit card or prepaid ca

In [8]:
SYSTEM_PROMPT = "You are a support-ticket triage assistant. Respond with JSON only."


def build_user_prompt(text: str) -> str:
    return (
        "Classify this consumer complaint.\n\n"
        f"Allowed products: {json.dumps(PRODUCTS)}\n"
        f"Allowed issues: {json.dumps(ISSUES)}\n\n"
        'Return JSON with exactly two keys: "product" and "issue".\n\n'
        f"Complaint:\n{text}"
    )


def build_target(row: pd.Series) -> str:
    return json.dumps({"product": row["product"], "issue": row["issue"]})


sample = train_df.iloc[0]
print(build_user_prompt(sample["text"]))
print("\nTARGET:", build_target(sample))

Classify this consumer complaint.

Allowed products: ["Credit reporting", "Debt collection", "Credit card or prepaid card", "Checking or savings account", "Mortgage", "Money transfer or virtual currency"]
Allowed issues: ["Managing an account", "Trouble during payment process", "Attempts to collect debt not owed", "Incorrect information on your report", "Problem with a credit reporting company's investigation into an existing problem", "Fraud or scam", "Improper use of your report", "Problem with a purchase shown on your statement", "Struggling to pay mortgage", "Other transaction problem", "Written notification about debt", "Problem with a lender or other company charging your account", "Other"]

Return JSON with exactly two keys: "product" and "issue".

Complaint:
My XXXX XXXX card handled by syncrony bank has a good cash back program- this is the second time in less than 2 mo that my rewards havent been accurate and actually have been altered- the compakhy tells me its a computer is

In [11]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

JSON_PATTERN = re.compile(r"\{.*?\}", re.DOTALL)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16,
)


def render_prompt(text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(text)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


@torch.inference_mode()
def generate_batch(
    m, texts: list[str], max_new_tokens: int = 96, batch_size: int = 16
) -> list[str]:
    m.eval()
    outputs: list[str] = []
    for start in range(0, len(texts), batch_size):
        prompts = [render_prompt(t) for t in texts[start : start + batch_size]]
        enc = tokenizer(
            prompts, return_tensors="pt", padding=True, add_special_tokens=False
        ).to(m.device)
        generated = m.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        outputs.extend(
            tokenizer.batch_decode(
                generated[:, enc["input_ids"].shape[1] :], skip_special_tokens=True
            )
        )
    return outputs


def parse_prediction(text: str) -> dict | None:
    match = JSON_PATTERN.search(text)
    if match is None:
        return None
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return parsed if isinstance(parsed, dict) else None


def evaluate(
    outputs: list[str], gold: pd.DataFrame, elapsed: float = float("nan")
) -> dict[str, float]:
    parsed = [parse_prediction(o) for o in outputs]
    pred_product = [str((p or {}).get("product", "__invalid__")) for p in parsed]
    pred_issue = [str((p or {}).get("issue", "__invalid__")) for p in parsed]
    gold_product = gold["product"].tolist()
    gold_issue = gold["issue"].tolist()

    valid_json = [p is not None for p in parsed]
    schema_valid = [
        p is not None and p.get("product") in PRODUCTS and p.get("issue") in ISSUES
        for p in parsed
    ]
    exact = [
        gp == pp and gi == pi
        for gp, pp, gi, pi in zip(gold_product, pred_product, gold_issue, pred_issue)
    ]

    return {
        "valid_json_%": round(float(100 * np.mean(valid_json)), 1),
        "schema_valid_%": round(float(100 * np.mean(schema_valid)), 1),
        "product_acc_%": round(100 * accuracy_score(gold_product, pred_product), 1),
        "product_macro_f1": round(
            f1_score(gold_product, pred_product, labels=PRODUCTS, average="macro", zero_division=0), 3
        ),
        "issue_acc_%": round(100 * accuracy_score(gold_issue, pred_issue), 1),
        "issue_macro_f1": round(
            f1_score(gold_issue, pred_issue, labels=ISSUES, average="macro", zero_division=0), 3
        ),
        "exact_match_%": round(float(100 * np.mean(exact)), 1),
        "sec_per_example": round(elapsed / len(outputs), 3),
    }


majority_output = json.dumps(
    {"product": train_df["product"].mode()[0], "issue": train_df["issue"].mode()[0]}
)
res_majority = evaluate([majority_output] * len(test_df), test_df)
print("Majority baseline:", res_majority)
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Majority baseline: {'valid_json_%': 100.0, 'schema_valid_%': 100.0, 'product_acc_%': 19.0, 'product_macro_f1': 0.053, 'issue_acc_%': 29.7, 'issue_macro_f1': 0.035, 'exact_match_%': 1.0, 'sec_per_example': nan}
GPU memory used (GB): 1.15


In [12]:
start = time.perf_counter()
base_outputs = generate_batch(model, test_df["text"].tolist())
base_elapsed = time.perf_counter() - start

res_base = evaluate(base_outputs, test_df, base_elapsed)
print("Prompted base model:", res_base)

for raw_output in base_outputs[:3]:
    print("-", raw_output[:200])

Prompted base model: {'valid_json_%': 100.0, 'schema_valid_%': 91.0, 'product_acc_%': 50.0, 'product_macro_f1': 0.474, 'issue_acc_%': 14.0, 'issue_macro_f1': 0.107, 'exact_match_%': 8.0, 'sec_per_example': 0.11}
- ```json
{
  "product": "Credit card or prepaid card",
  "issue": "Attempts to collect debt not owed"
}
```
- ```json
{
  "product": "Mortgage",
  "issue": "Problem with a purchase shown on your statement"
}
```
- ```json
{
  "product": "Credit card or prepaid card",
  "issue": "Attempts to collect debt not owed"
}
```


In [13]:
from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/ticket-triage-qlora/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(os.listdir(CHECKPOINT_DIR))

Mounted at /content/drive
[]


In [15]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

LORA_TARGET_MODULES = (
    "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
)


def encode_example(row: pd.Series) -> dict[str, list[int]]:
    prompt_ids = tokenizer(render_prompt(row["text"]), add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(
        build_target(row) + tokenizer.eos_token, add_special_tokens=False
    )["input_ids"]
    input_ids = prompt_ids + target_ids
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + target_ids,
    }


def build_dataset(df: pd.DataFrame, max_len: int) -> Dataset:
    encoded = [encode_example(row) for _, row in df.iterrows()]
    kept = [ex for ex in encoded if len(ex["input_ids"]) <= max_len]
    print(f"Kept {len(kept)}/{len(encoded)} examples (max_len={max_len})")
    return Dataset.from_list(kept)


def collate_batch(batch: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    longest = max(len(ex["input_ids"]) for ex in batch)

    def pad(key: str, value: int) -> torch.Tensor:
        return torch.tensor(
            [ex[key] + [value] * (longest - len(ex[key])) for ex in batch]
        )

    return {
        "input_ids": pad("input_ids", tokenizer.pad_token_id),
        "attention_mask": pad("attention_mask", 0),
        "labels": pad("labels", -100),
    }


def latest_checkpoint(directory: str) -> str | None:
    names = [d for d in os.listdir(directory) if d.startswith("checkpoint-")]
    if not names:
        return None
    newest = max(names, key=lambda d: int(d.split("-")[1]))
    return os.path.join(directory, newest)


lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=2 * LORA_R,
    lora_dropout=0.05,
    target_modules=list(LORA_TARGET_MODULES),
    task_type="CAUSAL_LM",
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_ds = build_dataset(train_df, MAX_LEN)
val_ds = build_dataset(val_df, MAX_LEN)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)

model.config.use_cache = False
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_batch,
)

resume_path = latest_checkpoint(CHECKPOINT_DIR)
print("Resuming from:", resume_path)
trainer.train(resume_from_checkpoint=resume_path)
model.config.use_cache = True

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Kept 3000/3000 examples (max_len=900)
Kept 200/200 examples (max_len=900)
Resuming from: None


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.074909,0.069459
2,0.046831,0.062681


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

In [16]:
ADAPTER_DIR = "/content/drive/MyDrive/ticket-triage-qlora/adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved:", os.listdir(ADAPTER_DIR))

start = time.perf_counter()
ft_outputs = generate_batch(model, test_df["text"].tolist())
ft_elapsed = time.perf_counter() - start
res_ft = evaluate(ft_outputs, test_df, ft_elapsed)

results = pd.DataFrame(
    {
        "Majority guess": res_majority,
        "Prompted base": res_base,
        "Fine-tuned (QLoRA)": res_ft,
    }
)
print(results.to_string())

for raw_output in ft_outputs[:3]:
    print("-", raw_output[:200])

Adapter saved: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']
                  Majority guess  Prompted base  Fine-tuned (QLoRA)
valid_json_%             100.000        100.000             100.000
schema_valid_%           100.000         91.000              99.700
product_acc_%             19.000         50.000              84.000
product_macro_f1           0.053          0.474               0.837
issue_acc_%               29.700         14.000              60.000
issue_macro_f1             0.035          0.107               0.484
exact_match_%              1.000          8.000              58.000
sec_per_example              NaN          0.110               0.172
- {"product": "Checking or savings account", "issue": "Managing an account"}
- {"product": "Mortgage", "issue": "Other"}
- {"product": "Debt collection", "issue": "Attempts to collect debt not owed"}


In [17]:
RESULTS_DIR = "/content/drive/MyDrive/ticket-triage-qlora/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

results.to_csv(f"{RESULTS_DIR}/metrics.csv")

predictions = test_df.assign(base_output=base_outputs, ft_output=ft_outputs)
predictions["ft_product"] = [(parse_prediction(o) or {}).get("product") for o in ft_outputs]
predictions["ft_issue"] = [(parse_prediction(o) or {}).get("issue") for o in ft_outputs]
predictions.to_csv(f"{RESULTS_DIR}/test_predictions.csv", index=False)

product_confusion = pd.crosstab(
    predictions["product"],
    predictions["ft_product"].fillna("invalid"),
    rownames=["gold"],
    colnames=["predicted"],
)
print(product_confusion.to_string())

issue_recall = (
    predictions.assign(hit=predictions["issue"] == predictions["ft_issue"])
    .groupby("issue")["hit"]
    .agg(["mean", "size"])
    .rename(columns={"mean": "recall", "size": "n"})
    .sort_values("recall")
    .round(2)
)
print(issue_recall.to_string())

predicted                           Checking or savings account  Credit card or prepaid card  Credit reporting  Debt collection  Health insurance  Money transfer or virtual currency  Mortgage
gold                                                                                                                                                                                           
Checking or savings account                                  30                            3                 0                1                 0                                   6         0
Credit card or prepaid card                                   3                           37                 4                1                 0                                   4         0
Credit reporting                                              0                            3                50                2                 0                                   0         2
Debt collection                         

In [18]:
RARE_ISSUES = (
    "Other transaction problem",
    "Problem with a lender or other company charging your account",
    "Written notification about debt",
)

print("Training examples per rare issue:")
print(train_df["issue"].value_counts().reindex(list(RARE_ISSUES)).to_string())

rare = predictions[predictions["issue"].isin(RARE_ISSUES)]
confusion = pd.crosstab(
    rare["ft_issue"].fillna("invalid"),
    rare["issue"],
    rownames=["predicted"],
    colnames=["gold"],
)
print(confusion.to_string())

Training examples per rare issue:
issue
Other transaction problem                                       98
Problem with a lender or other company charging your account    79
Written notification about debt                                 66
gold                                             Other transaction problem  Problem with a lender or other company charging your account  Written notification about debt
predicted                                                                                                                                                                
Attempts to collect debt not owed                                        0                                                             0                                5
Fraud or scam                                                            1                                                             0                                0
Managing an account                                                      0     